# ASCA AI - Market Scout Agent Testing Notebook

This notebook tests **Market Scout Agent** (`src/agents/market_scout.py`) and its **Time-Series Prophet Forecasting Engine** (`src/agents/tools/forecasting_tool.py`).
It demonstrates **asynchronous parallel execution (`asyncio.gather`)**, semaphore concurrency limits (4 max concurrent tasks), and surplus anomaly detection (>25% price drop) for Dambulla and Thambuththegama.

In [1]:
import sys
import asyncio
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.agents.market_scout import market_scout_agent
from src.infrastructure.db import init_db

print("Market Scout Agent modules loaded successfully!")

d:\1.Education\My Projects\ASCA AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


Market Scout Agent modules loaded successfully!


## Step 1: Initialize Async Database & Test Single Crop Forecasting

In [2]:
await init_db()

print("Running 14-day Prophet forecast for Dambulla Tomato...")
insight = await market_scout_agent.analyze_crop_async("DAMBULLA", "tomato")

print(f"\n--- SINGLE CROP FORECAST RESULT ---")
print(f"Center: {insight.center_id}")
print(f"Crop: {insight.crop_name}")
print(f"Current Wholesale Price: LKR {insight.current_wholesale_price_lkr:.2f}")
print(f"Predicted 14-Day Price: LKR {insight.predicted_wholesale_price_lkr:.2f}")
print(f"Supply Volume: {insight.supply_volume_tons} Metric Tons")
print(f"Surplus Anomaly Detected: {insight.surplus_anomaly_detected}")
print(f"Risk Level: {insight.risk_level.value}")

2026-07-30 12:32:06 | INFO     | src.infrastructure.db:init_db:22 - Initializing Database Tables...
2026-07-30 12:32:06 | INFO     | src.infrastructure.db:init_db:25 - Database Tables Initialized successfully.
Running 14-day Prophet forecast for Dambulla Tomato...
2026-07-30 12:32:06 | WARNING  | src.agents.market_scout:fetch_historical_data_async:58 - Insufficient DB records for tomato at DAMBULLA. Using synthetic data.


12:32:07 - cmdstanpy - INFO - Chain [1] start processing
12:32:08 - cmdstanpy - INFO - Chain [1] done processing



--- SINGLE CROP FORECAST RESULT ---
Center: DAMBULLA
Crop: tomato
Current Wholesale Price: LKR 169.11
Predicted 14-Day Price: LKR 168.94
Supply Volume: 56.6 Metric Tons
Surplus Anomaly Detected: False
Risk Level: LOW


## Step 2: Parallel Crop Scouting with Semaphore Throttling (Dambulla)

In [3]:
target_crops = ["tomato", "carrot", "beans", "eggplant", "cabbage", "green_chilli"]
print(f"🚀 MarketScoutAgent starting parallel forecasting for DAMBULLA ({len(target_crops)} crops)...\n")

insights = await market_scout_agent.scout_market_parallel_async(
    center_id="DAMBULLA",
    crops=target_crops
)

print(f"\n--- DAMBULLA MARKET SCOUT RESULTS ({len(insights)} Crops Analyzed) ---")
for item in insights:
    status_flag = "⚠️ SURPLUS ANOMALY" if item.surplus_anomaly_detected else "NORMAL"
    print(f"Crop: {item.crop_name:<12} | Current: LKR {item.current_wholesale_price_lkr:<6.2f} | Predicted: LKR {item.predicted_wholesale_price_lkr:<6.2f} | Anomaly: {status_flag:<18} | Risk: {item.risk_level.value}")

🚀 MarketScoutAgent starting parallel forecasting for DAMBULLA (6 crops)...

2026-07-30 12:32:31 | INFO     | src.agents.market_scout:scout_market_parallel_async:114 - MarketScoutAgent scouting 6 crops concurrently for DAMBULLA...
2026-07-30 12:32:31 | WARNING  | src.agents.market_scout:fetch_historical_data_async:58 - Insufficient DB records for tomato at DAMBULLA. Using synthetic data.
2026-07-30 12:32:31 | WARNING  | src.agents.market_scout:fetch_historical_data_async:58 - Insufficient DB records for eggplant at DAMBULLA. Using synthetic data.
2026-07-30 12:32:31 | WARNING  | src.agents.market_scout:fetch_historical_data_async:58 - Insufficient DB records for beans at DAMBULLA. Using synthetic data.
2026-07-30 12:32:31 | WARNING  | src.agents.market_scout:fetch_historical_data_async:58 - Insufficient DB records for carrot at DAMBULLA. Using synthetic data.


12:32:32 - cmdstanpy - INFO - Chain [1] start processing
12:32:32 - cmdstanpy - INFO - Chain [1] start processing
12:32:32 - cmdstanpy - INFO - Chain [1] start processing
12:32:32 - cmdstanpy - INFO - Chain [1] start processing
12:32:35 - cmdstanpy - INFO - Chain [1] done processing
12:32:35 - cmdstanpy - INFO - Chain [1] done processing


2026-07-30 12:32:35 | WARNING  | src.agents.market_scout:fetch_historical_data_async:58 - Insufficient DB records for cabbage at DAMBULLA. Using synthetic data.
2026-07-30 12:32:35 | WARNING  | src.agents.market_scout:fetch_historical_data_async:58 - Insufficient DB records for green_chilli at DAMBULLA. Using synthetic data.


12:32:36 - cmdstanpy - INFO - Chain [1] done processing
12:32:36 - cmdstanpy - INFO - Chain [1] start processing
12:32:36 - cmdstanpy - INFO - Chain [1] start processing
12:32:37 - cmdstanpy - INFO - Chain [1] done processing
12:32:38 - cmdstanpy - INFO - Chain [1] done processing
12:32:39 - cmdstanpy - INFO - Chain [1] done processing


2026-07-30 12:32:39 | INFO     | src.agents.market_scout:scout_market_parallel_async:127 - MarketScoutAgent completed scouting for DAMBULLA: Detected 0 surplus anomalies.

--- DAMBULLA MARKET SCOUT RESULTS (6 Crops Analyzed) ---
Crop: tomato       | Current: LKR 180.28 | Predicted: LKR 168.26 | Anomaly: NORMAL             | Risk: LOW
Crop: carrot       | Current: LKR 241.07 | Predicted: LKR 226.62 | Anomaly: NORMAL             | Risk: LOW
Crop: beans        | Current: LKR 199.37 | Predicted: LKR 196.09 | Anomaly: NORMAL             | Risk: LOW
Crop: eggplant     | Current: LKR 145.84 | Predicted: LKR 140.35 | Anomaly: NORMAL             | Risk: LOW
Crop: cabbage      | Current: LKR 131.83 | Predicted: LKR 116.79 | Anomaly: NORMAL             | Risk: MEDIUM
Crop: green_chilli | Current: LKR 330.52 | Predicted: LKR 318.78 | Anomaly: NORMAL             | Risk: LOW


## Step 3: Concurrent Multi-Center Scouting (Dambulla & Thambuththegama)

In [4]:
print("🚀 Running MarketScoutAgent for DAMBULLA and THAMBUTHTHEGAMA concurrently in parallel...\n")

dambulla_task = market_scout_agent.scout_market_parallel_async("DAMBULLA", ["tomato", "carrot", "beans"])
thambuththegama_task = market_scout_agent.scout_market_parallel_async("THAMBUTHTHEGAMA", ["tomato", "carrot", "beans"])

dambulla_insights, thambuththegama_insights = await asyncio.gather(dambulla_task, thambuththegama_task)

print(f"\n--- CONCURRENT MULTI-CENTER SCOUTING SUMMARY ---")
print(f"Dambulla Insights Count: {len(dambulla_insights)} | Anomalies: {sum(1 for i in dambulla_insights if i.surplus_anomaly_detected)}")
print(f"Thambuththegama Insights Count: {len(thambuththegama_insights)} | Anomalies: {sum(1 for i in thambuththegama_insights if i.surplus_anomaly_detected)}")

🚀 Running MarketScoutAgent for DAMBULLA and THAMBUTHTHEGAMA concurrently in parallel...

2026-07-30 12:33:12 | INFO     | src.agents.market_scout:scout_market_parallel_async:114 - MarketScoutAgent scouting 3 crops concurrently for DAMBULLA...
2026-07-30 12:33:12 | INFO     | src.agents.market_scout:scout_market_parallel_async:114 - MarketScoutAgent scouting 3 crops concurrently for THAMBUTHTHEGAMA...
2026-07-30 12:33:12 | WARNING  | src.agents.market_scout:fetch_historical_data_async:58 - Insufficient DB records for tomato at THAMBUTHTHEGAMA. Using synthetic data.
2026-07-30 12:33:12 | WARNING  | src.agents.market_scout:fetch_historical_data_async:58 - Insufficient DB records for tomato at DAMBULLA. Using synthetic data.
2026-07-30 12:33:12 | WARNING  | src.agents.market_scout:fetch_historical_data_async:58 - Insufficient DB records for beans at DAMBULLA. Using synthetic data.
2026-07-30 12:33:12 | WARNING  | src.agents.market_scout:fetch_historical_data_async:58 - Insufficient DB reco

12:33:13 - cmdstanpy - INFO - Chain [1] start processing
12:33:13 - cmdstanpy - INFO - Chain [1] start processing
12:33:13 - cmdstanpy - INFO - Chain [1] start processing
12:33:13 - cmdstanpy - INFO - Chain [1] start processing
12:33:13 - cmdstanpy - INFO - Chain [1] start processing
12:33:13 - cmdstanpy - INFO - Chain [1] start processing
12:33:18 - cmdstanpy - INFO - Chain [1] done processing
12:33:18 - cmdstanpy - INFO - Chain [1] done processing
12:33:18 - cmdstanpy - INFO - Chain [1] done processing
12:33:18 - cmdstanpy - INFO - Chain [1] done processing
12:33:18 - cmdstanpy - INFO - Chain [1] done processing
12:33:18 - cmdstanpy - INFO - Chain [1] done processing


2026-07-30 12:33:19 | INFO     | src.agents.market_scout:scout_market_parallel_async:127 - MarketScoutAgent completed scouting for DAMBULLA: Detected 0 surplus anomalies.
2026-07-30 12:33:19 | INFO     | src.agents.market_scout:scout_market_parallel_async:127 - MarketScoutAgent completed scouting for THAMBUTHTHEGAMA: Detected 0 surplus anomalies.

--- CONCURRENT MULTI-CENTER SCOUTING SUMMARY ---
Dambulla Insights Count: 3 | Anomalies: 0
Thambuththegama Insights Count: 3 | Anomalies: 0
